# Import necessary python libs +  Keras + Sklearn

In [36]:
import os
import tensorflow
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib as plt
import plotly.express as px


from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Activation, InputLayer, Dropout
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping
from keras.regularizers import l1, l2

In [ ]:
import random

def plot_convergence(hist, metric): 
    df = pd.DataFrame(hist.history)
    fig = px.line(df,
                x=np.arange(df[metric].shape[0]),
                y=[f'{metric}', f'val_{metric}'])
    fig.show()

def reset_seeds():
    os.environ['PYTHONHASHSEED']=str(42)
    tf.random.set_seed(42)
    np.random.seed(42)
    random.seed(42)

# Reading Dataset

In [38]:
data = pd.read_csv('https://raw.githubusercontent.com/renansantosmendes/lectures-cdas-2023/master/fetal_health.csv')
data.head()

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,mean_value_of_long_term_variability,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,2.4,64.0,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,10.4,130.0,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,13.4,130.0,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,23.0,117.0,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,19.9,117.0,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


# Clean up data and standardize dataset

In [39]:
## Remove class that we want to answer on this execise
x = data.drop(["fetal_health"], axis = 1)
y = data["fetal_health"] - 1

columns_names = list(x.columns)
x_df = preprocessing.StandardScaler().fit_transform(x) 
x_df = pd.DataFrame(x_df, columns=columns_names)

#devide dataset for training / test
x_train, X_test, y_train, Y_test = train_test_split(x_df, y, test_size=0.3, random_state=42)

#Keras understand that first class is 0
y_train = keras.utils.to_categorical(y_train)
Y_test = keras.utils.to_categorical(Y_test)

In [40]:
x_train.head()

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,mean_value_of_long_term_variability,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
1718,-0.234167,1.247640,-0.203210,-0.803434,-0.300544,-0.057476,-0.268754,0.407817,0.189365,-0.535361,-0.513181,0.399370,-0.357981,0.277292,0.994270,-0.458444,0.827234,0.473990,0.616025,0.110177,1.112980
857,0.883886,-0.822388,-0.203210,-0.124404,-0.638438,-0.057476,-0.268754,-0.173958,-0.603357,-0.480991,0.517578,-0.987146,1.097020,-0.335865,-1.040530,-0.458444,0.460877,0.730565,0.616025,-0.580173,-0.524526
1075,0.375680,-0.304881,-0.203210,0.894142,1.388924,-0.057476,-0.268754,-1.162976,0.302611,-0.535361,0.339861,0.270989,-0.256469,0.165809,-0.701397,-0.458444,0.399817,0.089126,0.201179,0.144694,1.112980
371,0.477322,-0.822388,-0.010304,-0.803434,0.037349,-0.057476,-0.268754,0.465995,-0.716603,-0.535361,0.197687,-0.576326,0.318764,-0.726055,-0.023130,-0.458444,0.399817,0.345702,0.339461,-0.476621,1.112980
222,-0.437449,-0.304881,0.075432,-1.482465,-0.638438,-0.057476,-0.268754,0.000575,-0.490111,-0.535361,1.619423,0.861543,-1.406934,-0.447348,0.655137,-0.458444,-0.271839,-0.103306,-0.282808,-0.511138,1.112980


#Creating Model and Adding the Layears

In [41]:
reset_seeds()
data_shape = x_train.iloc[0].shape

# WHAT is Sequential
# Sequential is the simplest way to build a neural network in Keras. Uses when no complex connections. Other types are Functional API and SubClassing

model = Sequential()

# Addign input layer
model.add(InputLayer(shape=data_shape))

# WHAT IS DENSE?
# Dense is a layer that is a fully connect Neural Network Layer
# Performs linear combination + activation
#Adding 4 Hidden Layers with 30 Neurons
model.add(Dense(units=30, activation='relu'))
model.add(Dense(units=30, activation='relu'))
model.add(Dense(units=30, activation='relu'))
model.add(Dense(units=30, activation='relu'))

#Adding Output Layer
#Softmax converts the outputs into probability
model.add(Dense(units=3, activation='softmax'))


In [42]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                │ (None, 30)             │           660 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 30)             │           930 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 30)             │           930 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 30)             │           930 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 3)              │            93 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,543 (13.84 KB)

 Trainable params: 3,543 (13.84 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [44]:
%%time
hist = model.fit(x = x_train, y = y_train, epochs = 30, batch_size = 128, validation_split = 0.2)


Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.5261 - loss: 1.0427 - val_accuracy: 0.7819 - val_loss: 0.9003
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7681 - loss: 0.8523 - val_accuracy: 0.8054 - val_loss: 0.7581
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7723 - loss: 0.7340 - val_accuracy: 0.8054 - val_loss: 0.6676
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7731 - loss: 0.6480 - val_accuracy: 0.8255 - val_loss: 0.5925
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8042 - loss: 0.5743 - val_accuracy: 0.8356 - val_loss: 0.5272
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8244 - loss: 0.5140 - val_accuracy: 0.8423 - val_loss: 0.4743
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8269 - loss: 0.4680 - val_accuracy: 0.8423 - val_loss: 0.4331
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8303 - loss: 0.4323 - val_accuracy: 0.8356 - v

# Evaluating the Model

In [45]:
plot_convergence(hist, 'loss')